# 第 14 週 實作｜一階可分離變數微分方程

整學期都在「給函數、求導數」。今天反過來:給一個關於導數的方程式,把函數找出來。這也是 capstone 的第一塊基石。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜方向場:不解方程也看得見解

觀念 2 說方向場能讓你「不解方程就看出行為」。這格畫三個方向場,並把真正的解曲線疊上去對照。


In [ ]:
def slope_field(ax, f, trange, yrange, title, n=18):
    T, Y = np.meshgrid(np.linspace(*trange, n), np.linspace(*yrange, n))
    S = f(T, Y)
    # 把每根小線段正規化成等長,只看方向
    L = np.hypot(1, S)
    ax.quiver(T, Y, 1/L, S/L, angles='xy', width=0.003, color='0.6')
    ax.set_title(title, fontsize=10); ax.set_xlabel('t'); ax.set_ylabel('y')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# (1) y' = y  —— 指數
slope_field(axes[0], lambda T, Y: Y, (0, 2), (-2, 4), "y' = y")
for c in [-1, -0.3, 0.3, 1]:
    ts = np.linspace(0, 2, 100); axes[0].plot(ts, c*np.exp(ts), lw=1.5)

# (2) y' = -y  —— 衰減,y=0 穩定
slope_field(axes[1], lambda T, Y: -Y, (0, 2), (-2, 2), "y' = -y   (y=0 stable)")
for c in [-1.5, -0.5, 0.5, 1.5]:
    ts = np.linspace(0, 2, 100); axes[1].plot(ts, c*np.exp(-ts), lw=1.5)

# (3) y' = y(1-y) —— 邏輯斯
slope_field(axes[2], lambda T, Y: Y*(1-Y), (0, 6), (-0.4, 1.6),
            "y' = y(1-y)   (y=1 stable, y=0 unstable)")
for y0 in [0.05, 0.3, 0.7, 1.4]:
    ts = np.linspace(0, 6, 200)
    A = (1-y0)/y0
    axes[2].plot(ts, 1/(1 + A*np.exp(-ts)), lw=1.5)
axes[2].axhline(1, color='C3', ls='--', lw=1); axes[2].axhline(0, color='C3', ls=':', lw=1)

plt.tight_layout(); plt.show()

print("平衡解與穩定性(用符號分析,不解方程):")
for name, f, eqs in [("y' = y", lambda y: y, [0]),
                     ("y' = -y", lambda y: -y, [0]),
                     ("y' = y(1-y)", lambda y: y*(1-y), [0, 1])]:
    print(f"\n  {name}")
    for e in eqs:
        left, right = f(e - 0.1), f(e + 0.1)
        stable = left > 0 and right < 0
        print(f"    y={e}: 左側 y'={left:+.3f}, 右側 y'={right:+.3f}  → "
              f"{'穩定(兩側都被吸引)' if stable else '不穩定'}")

In [ ]:
# TODO 學生練習:畫 y' = (y-2)*(y+1) 的方向場
# 先用符號分析預測 y=2 和 y=-1 哪個穩定,再看圖驗證

## Lab 2｜手刻 Euler 法,量它的一階收斂

觀念 7、8 說 Euler 是一階方法。這格把它寫出來(<strong>W16、W17 會直接重用這支函式</strong>),並量出那條斜率 1 的誤差線。


In [ ]:
def euler(f, y0, t0, t1, h):
    """顯式尤拉法解 y' = f(t, y)。回傳 (ts, ys) 兩個陣列。

    這支函式 W16(梯度流)與 W17(capstone)會直接重用。
    """
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        ys.append(y + h * f(t, y))
        ts.append(t + h)
    return np.array(ts), np.array(ys)

# 測試:y' = y, y(0) = 1 → y(t) = e^t
f = lambda t, y: y
exact = math.e

print(f"{'h':>9} {'步數':>6} {'Euler y(1)':>14} {'誤差':>12} {'誤差比值':>10}")
prev = None
hs, errs = [], []
for k in range(1, 8):
    h = 0.5**k
    ts, ys = euler(f, 1.0, 0.0, 1.0, h)
    e = abs(ys[-1] - exact)
    hs.append(h); errs.append(e)
    r = f"{prev/e:10.3f}" if prev else "         -"
    print(f"{h:9.5f} {len(ts)-1:6d} {ys[-1]:14.9f} {e:12.3e} {r}")
    prev = e

plt.loglog(hs, errs, 'o-', label='Euler')
plt.loglog(hs, np.array(hs)*errs[0]/hs[0], 'k--', label='slope 1 reference')
plt.xlabel('h'); plt.ylabel('|error at t=1|'); plt.legend()
plt.title("Euler's method is first order")
plt.show()

slope = np.polyfit(np.log10(hs), np.log10(errs), 1)[0]
print(f"\nlog-log 斜率 = {slope:.4f}   (理論 1.0)")
print(f"誤差比值趨近 2 → h 減半誤差減半 → O(h) ✓")

In [ ]:
# TODO 學生練習:改解 y' = -2y, y(0) = 1(精確解 e^(-2t))
# 試 h = 0.1 和 h = 1.5。後者會發生什麼事?為什麼?(提示:1 + h*(-2) 的絕對值)

## Lab 3｜三個模型:指數、冷卻、邏輯斯

觀念 4、5、6 的三個模型,把解析解與數值解畫在一起,並看 sigmoid 從哪裡冒出來。


In [ ]:
def euler(f, y0, t0, t1, h):
    ts, ys = [t0], [y0]
    while ts[-1] < t1 - 1e-12:
        t, y = ts[-1], ys[-1]
        ys.append(y + h * f(t, y)); ts.append(t + h)
    return np.array(ts), np.array(ys)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))

# (1) 指數衰減:半衰期與初始量無關
k = math.log(2) / 5                       # 半衰期 5 天
ts = np.linspace(0, 20, 200)
for y0 in [100, 60, 30]:
    ax[0].plot(ts, y0*np.exp(-k*ts), label=f'y0 = {y0}')
    ax[0].axhline(y0/2, ls=':', lw=0.7, color='0.7')
ax[0].axvline(5, color='C3', ls='--', label='half-life = 5')
ax[0].set_title('Decay: half-life is the same for every y0'); ax[0].legend(fontsize=8)

# (2) 牛頓冷卻:熱的降溫、冷的升溫,都趨向室溫
Ta, kc = 20, 0.0673
ts = np.linspace(0, 40, 200)
for T0 in [90, 60, 5]:
    ax[1].plot(ts, Ta + (T0-Ta)*np.exp(-kc*ts), label=f'T0 = {T0}')
ax[1].axhline(Ta, color='C3', ls='--', label='room 20')
ax[1].set_title('Cooling: same formula heats a cold drink'); ax[1].legend(fontsize=8)

# (3) 邏輯斯 = sigmoid,並和 Euler 數值解對照
logistic = lambda t, y: y*(1-y)
ts = np.linspace(-6, 6, 300)
ax[2].plot(ts, 1/(1+np.exp(-ts)), 'C0', lw=2, label='exact = sigmoid')
te, ye = euler(logistic, 1/(1+math.exp(6)), -6, 6, 0.25)
ax[2].plot(te, ye, 'C1.', ms=4, label='Euler h=0.25')
ax[2].axhline(1, color='C3', ls='--'); ax[2].axhline(0, color='C3', ls=':')
ax[2].set_title('Logistic solution IS the sigmoid'); ax[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

# 驗證 sigmoid' = sigmoid(1-sigmoid)
s = lambda z: 1/(1+math.exp(-z))
print("驗證 σ'(z) = σ(z)(1-σ(z)):")
for z in [-2, 0, 1, 3]:
    numeric = (s(z+1e-6) - s(z-1e-6)) / 2e-6
    formula = s(z)*(1-s(z))
    print(f"  z={z:3}:  數值微分 {numeric:.9f}   σ(1-σ) {formula:.9f}   "
          f"差 {abs(numeric-formula):.2e}")
print("\n→ sigmoid 的導數公式不是巧合,它就是邏輯斯方程本身")

In [ ]:
# TODO 學生練習:把邏輯斯改成有承載量 M=5 的版本 y' = y(1 - y/5)
# 解曲線會趨近多少?反曲點(成長最快)在 y = ? 用圖驗證